## 2026-1 M2177.003100: Deep Learning(001) - Coding Assignment

In this assignment, you will implement models using PyTorch.

If you are not familiar with PyTorch, we recommend referring to the official documentation:
 https://docs.pytorch.org/docs/stable/index.html

In [ ]:
!nvidia-smi

!pip uninstall torch torchvision torchaudio



'nvidia-smi'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

: 

## 1. ViT Implementation **(20 pts)**

### 1-1. Scaled Dot Product Attention **(Problem a, 5 pts)**

Implement the `forward` function of the Scaled Dot-Product Attention (SDPA).

The function takes query (Q), key (K), and value (V) as inputs and computes attention as follows:

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

---

You can follow the guidelines below:

- Compute attention scores using matrix multiplication between Q and K^T
- Scale the attention scores by sqrt(d_k)
- Apply softmax over the key dimension
- Multiply the attention weights with V to obtain the output

In [ ]:
class SDPA(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor
    ) -> torch.Tensor:
        """
        Q: (B, T_q, D_k)
        K: (B, T_k, D_k)
        V: (B, T_k, D_v)
        """
        ##### TODO 1 #####

        # 1. Compute attention scores: QK^T
        # 2. Scale by sqrt(d_k)
        # 3. Apply softmax over key dimension
        # 4. Compute weighted sum with V

        out =

        ##################

        return out

Use the following example to verify your implementation.

**Expected output:**
```python
tensor([[[0.4223, 0.1554, 0.4223],
         [0.3333, 0.3333, 0.3333],
         [0.2119, 0.5761, 0.2119]]])

In [ ]:
# input
X = torch.tensor([[[1.0, 0.0],
                   [0.0, 1.0],
                   [1.0, 1.0]]])   # (1, 3, 2)

# q, k, v
Q = torch.tensor([[[1.0], [0.0], [-1.0]]])   # (1, 3, 1)
K = torch.tensor([[[1.0], [0.0], [1.0]]])    # (1, 3, 1)
V = torch.tensor([[[1.0, 0.0, 0.0],
                   [0.0, 1.0, 0.0],
                   [0.0, 0.0, 1.0]]])        # (1, 3, 3)

# run SDPA
sdpa = SDPA()
out = sdpa(Q, K, V)

print(out)

### 1-2. Multihead Attention **(Problem b, 5 pts)**

Implement the `forward` function of the Multihead Attention (MHA).

The function takes an input tensor \( x \) and computes multi-head attention as follows:

MultiHead(Q, K, V) = Concat(head₁, ..., head_h) W_O  
where headᵢ = Attention(Qᵢ, Kᵢ, Vᵢ)

In this problem, assume that d_k = d_v = D / num_heads.

---

You can follow the guidelines below:

- Compute Q, K, V using linear projections from the input
- Split Q, K, V into multiple heads
- Apply scaled dot-product attention to each head
- Concatenate the outputs of all heads
- Apply a final linear projection to obtain the output

In [ ]:
class MHA(nn.Module):
    def __init__(self, dim: int = 16, num_heads: int = 1):
        super().__init__()
        assert dim % num_heads == 0

        self.num_heads = num_heads
        self.d_head = dim // num_heads  # assume d_k = d_v = d_head

        # projections
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        # output projection
        self.out_proj = nn.Linear(dim, dim)

        self.attn = SDPA()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        H = self.num_heads
        d = self.d_head

        ##### TODO 2 #####

        # 1. Project input to Q, K, V
        # 2. Split into multiple heads, (B, T, D) -> (B, H, T, d)
        # 3. Reshape for parallel attention (merge batch and heads)
        # 4. Apply scaled dot-product attention
        # 5. Restore original shape (concatenate heads), (B, H, T, d) -> (B*H, T, d)
        # 6. Apply final linear projection

        out =
        ##################

        return out

### 1-3. Patch Embedding **(Problem c, 4 pts)**

Implement the `__init__` function of the Patch Embedding module using a convolution layer.

The module splits the input image into non-overlapping patches and projects each patch into an embedding vector using convolution.

---

You can follow the guidelines below:

- Use a 2D convolution layer to extract patches from the image
- Set the kernel size equal to the patch size
- Set the stride equal to the patch size so that patches do not overlap
- The convolution should map from `in_channels` to `embed_dim`

---

After the convolution, the output should be reshaped in the `forward` function to:

(B, N, D), where N is the number of patches and D is the embedding dimension

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        in_channels: int = 3,
        embed_dim: int = 64
    ):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size

        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size

        ##### TODO 3 #####

        # Define a convolution layer that:
        # - uses kernel_size = patch_size
        # - uses stride = patch_size
        # - maps in_channels -> embed_dim

        self.proj =

        ##################

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)                 # (B, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2) # (B, N, D)
        return x

### 1-4. Positional Encoding **(Problem d, 6 pts)**

Implement the `__init__` function for different positional encodings used in Vision Transformers (ViT).

Each module should create a positional encoding in the `__init__` function and add it to the input in the `forward` function.

---

You can follow the guidelines below:

- For `PosEmbSinusoidal1D`, create 1D sinusoidal positional encodings based on sequence positions
- For `PosEmbSinusoidal2D`, create 2D sinusoidal positional encodings based on spatial positions (height and width)
- For `PosEmbLearnable`, define learnable positional embeddings as trainable parameters

---
Each positional encoding should have shape (1, N, D) and be added to the input tensor.

In [ ]:
import matplotlib.pyplot as plt

class PosEmb(nn.Module):
    def __init__(self, num_tokens: int = 16, dim: int = 4):
        super().__init__()
        self.num_tokens = num_tokens
        self.dim = dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_emb

    def visualize(self, channel: int = 0):
        """
        Visualize positional embedding as 2D image.
        """
        H = int(self.num_tokens ** 0.5)

        pe = self.pos_emb[0].detach()
        img = pe[:, channel].view(H, H).cpu()

        plt.imshow(img, cmap="viridis")
        plt.title(f"{self.__class__.__name__} (channel {channel})")
        plt.colorbar()
        plt.show()

##### (Problem d-1, 2 pts) **PosEmbSinusoidal1D**

  Create positional encodings using sine and cosine functions based on token positions.

  For each position `pos` from 0 to N-1 and dimension index `i` from 0 to D-1:

  - Use sine for even indices and cosine for odd indices:

    PE(pos, 2i)   = sin(pos / (10000^(2i / D)))  
    PE(pos, 2i+1) = cos(pos / (10000^(2i / D)))

  - Construct a tensor of shape (N, D), then unsqueeze to (1, N, D)

In [ ]:
class PosEmbSinusoidal1D(PosEmb):
    def __init__(self, num_tokens: int = 16, dim: int = 4):
        super().__init__(num_tokens, dim)

        ##### TODO 4 #####

        pos_emb =

        ##################

        self.register_buffer("pos_emb", pos_emb)

##### (Problem d-2, 2 pts) **PosEmbSinusoidal2D**

  Create positional encodings based on 2D spatial positions.

  - Let each token correspond to a grid position (y, x)
  - Split the embedding dimension into two halves:
    - First D/2 dimensions encode the row (y)
    - Second D/2 dimensions encode the column (x)

  - Apply the same sinusoidal formulation as in 1D separately for y and x:

    PE_y(y, :) and PE_x(x, :)

  - Concatenate them along the feature dimension:

    PE(y, x) = concat(PE_y, PE_x)

  - Flatten the grid to shape (N, D) where N = height × width, then unsqueeze to (1, N, D)


In [ ]:
class PosEmbSinusoidal2D(PosEmb):
    def __init__(self, num_tokens: int = 16, dim: int = 4):
        super().__init__(num_tokens, dim)

        H = int(num_tokens ** 0.5)
        assert H * H == num_tokens
        assert dim % 2 == 0

        ##### TODO 5 #####

        pos_emb =

        ##################

        self.register_buffer("pos_emb", pos_emb)

##### (Problem d-3, 2 pts) **PosEmbLearnable**

  Define a learnable positional embedding as a trainable parameter.

  - Initialize a tensor of shape (1, N, D)
  - Register it as an `nn.Parameter`
  - The values will be updated during training

In [ ]:
class PosEmbLearnable(PosEmb):
    def __init__(self, num_tokens: int = 16, dim: int = 4):
        super().__init__(num_tokens, dim)

        #### TODO 6 #####

        self.pos_emb =

        ##################

You can visualize the positional encodings to verify your implementation.

In [ ]:
pe1 = PosEmbSinusoidal1D(num_tokens=64*64, dim=16)
pe2 = PosEmbSinusoidal2D(num_tokens=64*64, dim=16)
pe3 = PosEmbLearnable(num_tokens=64*64, dim=16)

pe1.visualize()
pe2.visualize()
pe3.visualize()

### 1-5. Transformer Block **(Problem e, 0 pts)**

In this part, we provide an example implementation of a Transformer block used in Vision Transformers.

The block consists of two main components:
- Multi-head self-attention (MHA)
- Feed-forward network (MLP)

Each component is applied with layer normalization and residual connections.

---

You can follow the structure below:

- Apply layer normalization before the attention layer
- Add a residual connection after the attention layer
- Apply layer normalization before the MLP
- Add a residual connection after the MLP

---

The forward pass follows:

x = x + Attention(LayerNorm(x))  
x = x + MLP(LayerNorm(x))

In [ ]:
class Block(nn.Module):
    def __init__(self, embed_dim: int = 16, num_heads: int = 1, mlp_ratio: float = 1.0):
        super().__init__()

        # --- LayerNorm + Attention ---
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MHA(embed_dim, num_heads)

        # --- LayerNorm + MLP ---
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x):
        # Attention block
        x = x + self.attn(self.norm1(x))

        # MLP block
        x = x + self.fc2(self.act(self.fc1(self.norm2(x))))

        return x

### 1-6. ViT Architecture

In this part, we provide an example implementation of a Vision Transformer (ViT) architecture.

The model consists of the following components:
- Patch embedding
- Positional encoding
- Multiple Transformer blocks
- Final normalization and pooling

---

You can follow the structure below:

- Convert the input image into patch embeddings
- Add positional encodings to the patch embeddings
- Pass the sequence through multiple Transformer blocks
- Apply a final layer normalization
- Aggregate the sequence into a single representation using mean pooling

In [ ]:
class ViT(nn.Module):
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        embed_dim: int = 16,
        depth: int = 1,
        num_heads: int = 1,
        mlp_ratio: float = 1.0
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim
        )

        num_patches = self.patch_embed.num_patches

        # TODO: You can change positional encoding strategy if desired
        # self.pos_embed = PosEmbSinusoidal1D(num_tokens=num_patches, dim=embed_dim)
        self.pos_embed = PosEmbLearnable(num_tokens=num_patches, dim=embed_dim)

        self.blocks = nn.ModuleList(
            [Block(embed_dim, num_heads, mlp_ratio) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        x = self.pos_embed(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)

        return x.mean(dim=1)

## 2. Contrastive Learning **(30 pts)**

### 2-1. CLIP Loss Implementation **(Problem f, 4 pts)**

Implement the `clip_loss` function.

CLIP uses a symmetric contrastive loss between image and text embeddings.

Given image features and text features, compute a similarity matrix and apply cross-entropy loss in both directions.

---

You can follow the guidelines below:

- Normalize image and text features
- Compute similarity matrix using matrix multiplication
- Scale the logits using a temperature parameter (`logit_scale`)
- Construct labels as indices from 0 to B-1
- Compute cross-entropy loss for both image-to-text and text-to-image
- Return the average of the two losses

In [ ]:
def clip_loss(img_feat: torch.Tensor, txt_feat: torch.Tensor, logit_scale: torch.Tensor) -> torch.Tensor:
    img_feat = F.normalize(img_feat, dim=-1)
    txt_feat = F.normalize(txt_feat, dim=-1)

    ##### TODO 7 #####

    # 1. Compute pairwise similarity between image and text features
    # 2. Apply temperature scaling
    # 3. Use cross-entropy loss with matching indices as labels
    # 4. Compute loss in both directions (image-to-text and text-to-image)

    loss_i =
    loss_t =

    ##################

    return (loss_i + loss_t) / 2

### 2-2. SigLIP Loss Implementation **(Problem g, 4 pts)**

Implement the `siglip_loss` function.

SigLIP uses a sigmoid-based contrastive loss where each image-text pair is treated independently.

---

You can follow the guidelines below:

- Normalize image and text features
- Compute similarity matrix
- Scale the logits using a temperature parameter (`logit_scale`)
- Construct a label matrix where diagonal elements are positive pairs and others are negative
- Apply binary cross-entropy loss with logits
- Return the average loss

In [ ]:
def siglip_loss(img_feat: torch.Tensor, txt_feat: torch.Tensor, logit_scale: torch.Tensor):
    img_feat = F.normalize(img_feat, dim=-1)
    txt_feat = F.normalize(txt_feat, dim=-1)

    ##### TODO 8 #####

    # 1. Compute pairwise similarity between image and text features
    # 2. Apply temperature scaling
    # 3. Treat each image-text pair as a binary classification (match vs non-match)
    # 4. Use identity matrix as labels and apply BCE loss

    loss =

    ##################

    return loss

### 2-3. Train on COCO dataset **(Problem h, 20 pts)**

Data preparation

In [ ]:
# local machine
# !unzip -n data.zip
# root_path = "."

# colab
from google.colab import drive
drive.mount('/content/drive')

# modify path
root_path = "/content/drive/MyDrive/2026_sp_dl"
data_path = f"{root_path}/data.zip"

!unzip -n {data_path} -d {root_path}

In [ ]:
import os
import json
import math
import random
import torchvision.transforms as T

from PIL import Image
from tqdm import tqdm
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

#### Dataset class

In [ ]:
class CocoDataset(Dataset):
    def __init__(self, image_root, annotation_file, is_train=True):
        with open(annotation_file, "r") as f:
            data = json.load(f)

        self.images = data["images"]
        self.annotations = data["annotations"]
        self.image_root = image_root

        self.id_to_caps = {}
        for ann in self.annotations:
            self.id_to_caps.setdefault(ann["image_id"], []).append(ann["caption"])

        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        self.is_train = is_train


    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info["id"]

        img_path = os.path.join(self.image_root, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        if self.is_train:
            caption = random.choice(self.id_to_caps[img_id])
        else:
            caption = self.id_to_caps[img_id][0]
        return image, caption

You may add image augmentations.
Note that validation/test procedures must use the original transform function.

In [ ]:
train_dataset = CocoDataset(
    f"{root_path}/data/train/images",
    f"{root_path}/data/train/annotations/captions_train.json",
    is_train=True
)

valid_dataset = CocoDataset(
    f"{root_path}/data/valid/images",
    f"{root_path}/data/valid/annotations/captions_valid.json",
    is_train=False
)

# TODO: You may apply image augmentations during training
train_dataset.transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

#### Text encoder
We use pretrained DistilBERT text encoder for this assignment.

SANH, et al. DistilBERT, a distilled version of BERT: smaller,
faster, cheaper and lighter, EMC^2: 5th Edition Co-located with NeurIPS’19.

(https://arxiv.org/pdf/1910.01108)

In [ ]:
# Do not modify this block

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

for p in text_model.parameters():
    p.requires_grad = False

text_dim = text_model.config.hidden_size  # 768

#### Training

Using the ViT implemented above, perform contrastive training.

The following constraints must be strictly followed:
- Do not use any data other than the provided training dataset (e.g., validation set or external data).
- Do not use pretrained weights for the vision encoder.
- Do not train the provided text encoder.

Except for the above restrictions, all other design choices are free (e.g., model architecture (ViT design), number of training epochs, optimizer, learning rate scheduler, and image augmentations).

In [ ]:
vision_encoder = ViT(
    img_size=224,
).to(device)
vision_proj = nn.Linear(vision_encoder.embed_dim, text_dim).to(device)

logit_scale = nn.Parameter(
    torch.ones([], device=device) * math.log(1 / 0.07)
)

In [ ]:
TRAIN_BATCH_SIZE = 128
TRAIN_EPOCH = 10
SAVE_FREQ = 2
SERIAL = datetime.now().strftime("%y%m%d-%H%M%S")

In [ ]:
optimizer = torch.optim.SGD(
    list(vision_encoder.parameters()) +
    list(vision_proj.parameters()) +
    [logit_scale],
)

scheduler = torch.optim.lr_scheduler.ConstantLR(optimizer, factor=1.0, total_iters=TRAIN_EPOCH)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False
)

#### Evaluation

The following metrics are used to evaluate image-text retrieval performance:

- R@1/R@10 (Recall@1/Recall@10): The proportion of samples where the correct text is within the top 1/10 retrieved results. Higher is better.

- Mean/Median Rank: The average/median rank of the correct text across all samples. Lower is better.

- **Mean Top-% Position**: The average percentile position of the correct text among all candidates. Lower is better.

Grading will be based on the validation set **Mean Top-% Position** performance as follows:
- ≤ 20% → 5 pts  
- ≤ 15% → 10 pts  
- ≤ 12% → 15 pts  
- ≤ 10% → 20 pts  

However, if the evaluation on a hidden test set differs from the validation score by more than 50%, the test score will be used for grading.

In [ ]:
# Do not modify this block

@torch.no_grad()
def evaluate(loader, vision_encoder, vision_proj,
             text_model, tokenizer, device):

    vision_encoder.eval()
    vision_proj.eval()

    all_img = []
    all_txt = []

    for images, captions in loader:
        images = images.to(device)

        inputs = tokenizer(
            list(captions),
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        txt_out = text_model(**inputs)
        txt_feat = txt_out.last_hidden_state[:, 0]

        img_feat = vision_proj(vision_encoder(images))

        img_feat = F.normalize(img_feat, dim=-1)
        txt_feat = F.normalize(txt_feat, dim=-1)

        all_img.append(img_feat)
        all_txt.append(txt_feat)

    all_img = torch.cat(all_img)
    all_txt = torch.cat(all_txt)

    sim = all_img @ all_txt.T
    ranks = sim.argsort(dim=-1, descending=True)

    correct = torch.arange(len(sim), device=sim.device)

    r1 = (ranks[:, :1] == correct.unsqueeze(1)).any(dim=1).float().mean()
    r10 = (ranks[:, :10] == correct.unsqueeze(1)).any(dim=1).float().mean()

    correct_positions = (ranks == correct.unsqueeze(1)).nonzero(as_tuple=False)[:, 1]

    mean_rank = correct_positions.float().mean() + 1
    median_rank = correct_positions.float().median() + 1

    N = len(sim)
    percentiles = (correct_positions.float() + 1) / N * 100
    mean_percentile = percentiles.mean()

    print(f"Validation R@1:  {r1.item():.4f}")
    print(f"Validation R@10: {r10.item():.4f}")
    print(f"Mean Rank: {mean_rank.item():.2f}")
    print(f"Median Rank: {median_rank.item():.2f}")
    print(f"Mean Top-% Position: {mean_percentile.item():.2f}%")

    return mean_percentile.item()

#### Save/Load checkpoint

In [ ]:
def save_checkpoint(save_dir, vision_encoder, vision_proj):
    os.makedirs(save_dir, exist_ok=True)

    torch.save(vision_encoder.state_dict(), os.path.join(save_dir, "vision_encoder.pt"))
    torch.save(vision_proj.state_dict(), os.path.join(save_dir, "vision_proj.pt"))

    print(f"Saved checkpoint to {save_dir}")

def load_checkpoint(load_dir, vision_encoder, vision_proj, device="cpu"):
    ve_path = os.path.join(load_dir, "vision_encoder.pt")
    vp_path = os.path.join(load_dir, "vision_proj.pt")

    vision_encoder.load_state_dict(torch.load(ve_path, map_location=device))
    vision_proj.load_state_dict(torch.load(vp_path, map_location=device))

    vision_encoder.to(device)
    vision_proj.to(device)

    vision_encoder.eval()
    vision_proj.eval()

    print(f"Loaded checkpoint from {load_dir}")

In [ ]:
total_steps = len(train_loader) * TRAIN_EPOCH

best_score = evaluate(valid_loader, vision_encoder, vision_proj, text_model, tokenizer, device)
best_epoch = 0
val_history = []

for epoch in range(TRAIN_EPOCH):

        vision_encoder.train()
        vision_proj.train()

        total_loss = 0

        for images, captions in tqdm(train_loader):

            images = images.to(device)

            inputs = tokenizer(
                list(captions),
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():
                txt_out = text_model(**inputs)
                txt_feat = txt_out.last_hidden_state[:, 0]

            img_feat = vision_proj(vision_encoder(images))

            # TODO: You can choose your loss function between CLIP & SigLIP
            loss = clip_loss(img_feat, txt_feat, logit_scale)
            # loss = siglip_loss(img_feat, txt_feat, logit_scale)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                logit_scale.clamp_(max=math.log(100))

            total_loss += loss.item()

        print(
            f"\nEpoch {epoch} | "
            f"Loss: {total_loss/len(train_loader):.4f} | "
        )

        valid_score = evaluate(valid_loader, vision_encoder, vision_proj, text_model, tokenizer, device)
        val_history.append(valid_score)

        if (epoch + 1) % SAVE_FREQ == 0:
            save_dir = f"{root_path}/checkpoints/exp-{SERIAL}/epoch-{epoch + 1}_score-{valid_score:.3f}"
            save_checkpoint(save_dir, vision_encoder, vision_proj)
        if valid_score < best_score:
            save_dir = f"{root_path}/checkpoints/exp-{SERIAL}/best"
            save_checkpoint(save_dir, vision_encoder, vision_proj)

            best_epoch = epoch
            best_score = valid_score

In [ ]:
print(f"Best score: {best_score:.2f}%")

### 2-4. Visualization & Brief Discussion **(Problem i, 2 pts)**

Using the provided visualization code, plot the validation curve.

Based on the result, briefly describe your observations during training.

Discuss any challenges you encountered, hyperparameter choices, and attempts you made to improve the model performance.

Feel free to include additional qualitative analysis or visualization if helpful.

In [ ]:
epochs = list(range(len(val_history)))

plt.figure()
plt.plot(epochs, val_history, marker='o')

plt.xlabel("Epoch")
plt.ylabel("Validation Score")
plt.title("Validation Curve")

plt.grid()
plt.show()

### Answer:

## 3. AI Tool Usage & Reference

If you used any AI tools (e.g., ChatGPT) while working on this assignment, briefly describe how they were used.

Also, if you applied techniques not covered in class, please provide appropriate references.

### Answer:

#### Check if your checkpoint is loadable

In [ ]:
checkpoint_path = "path/to/checkpoint"

load_checkpoint(checkpoint_path, vision_encoder, vision_proj)